# Ron-0 Qwen LoRA training (Google Colab)

This notebook runs the repository's existing LoRA training and bilingual evaluation pipeline on a Colab GPU. It saves the adapter and report to Google Drive so they survive a temporary runtime. It does **not** change production weights or deploy anything.

Before running: in Colab choose **Runtime → Change runtime type → T4 GPU** (if available). Free GPU availability and session limits vary; if Colab gives no GPU, stop rather than running the real training on CPU.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU assigned. In Colab select Runtime → Change runtime type → T4 GPU, then rerun this cell.')


In [ ]:
from pathlib import Path
if not Path('/content/Ron-0/.git').exists():
    !git clone -b training/qwen-lora-finetune https://github.com/arkanws513-spec/Ron-0.git /content/Ron-0
%cd /content/Ron-0
!git pull --ff-only origin training/qwen-lora-finetune


In [ ]:
!python -m pip install -r requirements-training.txt


In [ ]:
!python scripts/download_oasst1_english.py


## Connect Google Drive

The output folder below is on your own Drive. This avoids losing adapter checkpoints when the Colab runtime disconnects. Approve the Google Drive mount prompt yourself.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUTPUT = Path('/content/drive/MyDrive/Ron0-qwen-lora-candidate')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Training artifacts will be saved to:', OUTPUT)


## Train an experimental adapter

The first run uses conservative settings. Training can take a while. The saved adapter is only a candidate; do not replace the live model unless the blind evaluation and the quantized-GGUF test both pass.


In [ ]:
!python scripts/train_qwen_lora.py --steps 500 --batch-size 1 --gradient-accumulation 8 --max-sequence-length 512 --eval-every 50 --output /content/drive/MyDrive/Ron0-qwen-lora-candidate


## Evaluate the adapter on the 300 bilingual prompts / 100 question families

This uses the same GPU and checks prompt-level and question-family pass rates. A failed gate means keep the existing production model and inspect the failures; it does not mean training is useless, but it is not ready for deployment.


In [ ]:
!python scripts/evaluate_qwen_lora.py --adapter /content/drive/MyDrive/Ron0-qwen-lora-candidate/adapter --cases training/qwen_eval_cases.json --report /content/drive/MyDrive/Ron0-qwen-lora-candidate/evaluation.json --min-pass-rate 0.80 --min-language-pass-rate 0.80


In [ ]:
import json
report_path = OUTPUT / 'evaluation.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({k: report.get(k) for k in [
        'passed', 'case_count', 'pass_rate', 'question_family_count',
        'question_family_pass_rate', 'critical_question_family_pass_rate',
        'question_family_language_metrics'
    ]}, ensure_ascii=False, indent=2))
else:
    print('No evaluation report yet. Run the evaluation cell after training finishes.')


## Important

- Keep `training_report.json`, `best_metrics.json`, and `evaluation.json` with the adapter.
- This notebook does not upload private data or model weights to Hugging Face Hub.
- Do not deploy based on training loss alone. First inspect the evaluation failures, merge the adapter, convert to GGUF, and repeat evaluation through Ron's actual `llama.cpp` runtime.
